# AdventureWorks — Exercise 2: Data Architecture & Schema Design

**Course:** Data Engineering  
**Notebook:** 02 — Schema Design, OLTP, OLAP, and Star Schema  
**Assistant:** Antigravity  

---

## What This Notebook Covers

| # | Section | Purpose |
|---|---------|----------|
| 1 | Setup & Load Raw Staging | Import modules, read Parquet from staging/raw/ |
| 2 | Data Validation | Schema, NULL, duplicate, and outlier checks |
| 3 | Data Transformation | Clean Product, Customer, and Sales tables |
| 4 | Aggregations | Sales by product, customer, date |
| 5 | OLTP Schema Design | Normalized operational model |
| 6 | Star Schema (OLAP) | DimDate, DimProduct, DimCustomer, FactSales |
| 7 | OLAP Operations | Roll-up, Drill-down, Slice, Dice |
| 8 | Summary | What was accomplished |

> **Full Refresh Architecture:** All data is rebuilt from source each run.


---
## Section 1 — Setup & Load from Raw Staging

**Objective:**  
Import all project modules and load the raw Parquet files written by Step 1.

> We read from `staging/raw/` rather than re-reading CSVs — the raw layer is the contract  
> between extraction and transformation. This is faster and preserves exact source state.


In [1]:
import sys
import pandas as pd
import numpy as np
from pathlib import Path

# Add src/ to path
NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from config import (
    STAGING_RAW, STAGING_VALID, STAGING_INVALID,
    STAGING_DUPLICATES, STAGING_OUTLIERS, ensure_directories,
)
from ingestion    import extract_all
from validation   import validate_all, print_validation_report
from transformation import transform_all
from olap import (
    build_dim_date, build_dim_product, build_dim_customer,
    build_fact_sales, build_olap,
    olap_rollup, olap_drilldown, olap_slice, olap_dice,
)

pd.set_option('display.max_columns', 20)
pd.set_option('display.float_format', '{:,.2f}'.format)
ensure_directories()

print('Modules loaded.')
print(f'Project Root: {PROJECT_ROOT}')

Modules loaded.
Project Root: F:\DE_CAT_1\AdventureWorks_DataEngineering


In [2]:
# ── Extract all four source tables ────────────────────────────
# Full Refresh: always read from the source CSV files
print('Extracting source data (Full Refresh) ...')
raw = extract_all()

print(f'\n  Product          : {len(raw["product"]):>7,} rows')
print(f'  Customer         : {len(raw["customer"]):>7,} rows')
print(f'  SalesOrderHeader : {len(raw["salesorderheader"]):>7,} rows')
print(f'  SalesOrderDetail : {len(raw["salesorderdetail"]):>7,} rows')
print('\nExtraction complete.')

Extracting source data (Full Refresh) ...

  Product          :     504 rows
  Customer         :  19,820 rows
  SalesOrderHeader :  31,465 rows
  SalesOrderDetail : 121,317 rows

Extraction complete.


**Explanation:**  
`extract_all()` reads all 4 source CSVs using the settings in `config.py`.  
This is Step 1 of our pipeline — every run starts fresh from the source files.


---
## Section 2 — Data Validation

**Objective:**  
Run four validation checks on all raw tables:
1. **Schema Validation** — are all expected columns present?
2. **Null Validation** — are key columns (ProductID, CustomerID, etc.) populated?
3. **Duplicate Detection** — are primary keys unique?
4. **Outlier Detection** — are numeric values within expected ranges (IQR method)?

**What is IQR?**  
The Inter-Quartile Range (IQR = Q3 − Q1) defines the "middle 50%" of data.  
Values below `Q1 − 1.5 × IQR` or above `Q3 + 1.5 × IQR` are flagged as outliers.


In [3]:
print('Running validation pipeline ...\n')
val_result = validate_all(raw)
print_validation_report(val_result)

Running validation pipeline ...


  Validating: product (504 records) ...
    [schema] OK
    [nulls]  valid=504  invalid=0
    [dupes]  unique=504  duplicates=0
    [outliers] 84 records flagged

  Validating: customer (19,820 records) ...
    [schema] OK
    [nulls]  valid=19,820  invalid=0
    [dupes]  unique=19,820  duplicates=0
    [outliers] 0 records flagged

  Validating: salesorderheader (31,465 records) ...
    [schema] OK
    [nulls]  valid=31,465  invalid=0
    [dupes]  unique=31,465  duplicates=0
    [outliers] 0 records flagged

  Validating: salesorderdetail (121,317 records) ...
    [schema] OK
    [nulls]  valid=121,317  invalid=0
    [dupes]  unique=121,317  duplicates=0
    [outliers] 18,852 records flagged

  VALIDATION REPORT

  TABLE: PRODUCT
    Schema   : OK
    Nulls    : 0 invalid  |  504 valid
    Dupes    : 0 duplicates  |  504 unique
    Outliers : 84 records flagged
      StandardCost: lo=-475.61  hi=792.69  outliers=55
      ListPrice: lo=-847.49  hi=1,41

In [4]:
# Inspect product outliers in detail
print('\n── Product Outlier Detail ──')
print('(Records where StandardCost, ListPrice, or Weight is outside IQR bounds)\n')

prod_valid = val_result['valid']['product']
if '_is_outlier' in prod_valid.columns:
    outliers = prod_valid[prod_valid['_is_outlier'] == True]
    show_cols = ['ProductID', 'Name', 'Color', 'StandardCost', 'ListPrice', 'Weight']
    available = [c for c in show_cols if c in outliers.columns]
    display(outliers[available].head(10))
    print(f'\nTotal outlier records: {len(outliers)}')
    print('Note: Outliers are flagged but NOT removed from the valid dataset.')
    print('They are saved separately in staging/outliers/')


── Product Outlier Detail ──
(Records where StandardCost, ListPrice, or Weight is outside IQR bounds)



,ProductID,Name,Color,StandardCost,ListPrice,Weight
179,507,LL Mountain Rim,NaN,0.00,0.00,435.00
180,508,ML Mountain Rim,NaN,0.00,0.00,450.00
181,509,HL Mountain Rim,NaN,0.00,0.00,400.00
182,510,LL Road Rim,NaN,0.00,0.00,445.00
183,511,ML Road Rim,NaN,0.00,0.00,450.00
184,512,HL Road Rim,NaN,0.00,0.00,400.00
185,513,Touring Rim,NaN,0.00,0.00,460.00
209,680,"HL Road Frame - Black, 58",Black,"1,059.31","1,431.50",2.24
210,706,"HL Road Frame - Red, 58",Red,"1,059.31","1,431.50",2.24
221,717,"HL Road Frame - Red, 62",Red,868.63,"1,431.50",2.30



Total outlier records: 84
Note: Outliers are flagged but NOT removed from the valid dataset.
They are saved separately in staging/outliers/


In [5]:
# Show what files were written to staging folders
print('\n── Staging Files Written by Validation ──\n')
for folder_name, folder_path in [
    ('valid/',      STAGING_VALID),
    ('invalid/',    STAGING_INVALID),
    ('duplicates/', STAGING_DUPLICATES),
    ('outliers/',   STAGING_OUTLIERS),
]:
    files = list(folder_path.glob('*.parquet'))
    print(f'  staging/{folder_name}')
    if files:
        for f in files:
            size_kb = round(f.stat().st_size / 1024, 1)
            print(f'    {f.name:<45} ({size_kb:>8.1f} KB)')
    else:
        print(f'    (empty — no records of this type)')
    print()


── Staging Files Written by Validation ──

  staging/valid/
    agg_avg_product_price.parquet                 (    13.7 KB)
    agg_sales_by_customer.parquet                 (   175.6 KB)
    agg_sales_by_date.parquet                     (    15.1 KB)
    agg_sales_by_month.parquet                    (     3.6 KB)
    agg_sales_by_product.parquet                  (    14.6 KB)
    agg_sales_by_year.parquet                     (     1.9 KB)
    agg_top10_products.parquet                    (     5.4 KB)
    transformed_customer.parquet                  (  1110.8 KB)
    transformed_product.parquet                   (    54.4 KB)
    transformed_sales.parquet                     (  2318.5 KB)
    valid_customer.parquet                        (  1119.5 KB)
    valid_product.parquet                         (    53.2 KB)
    valid_salesorderdetail.parquet                (  5904.7 KB)
    valid_salesorderheader.parquet                (  3109.3 KB)

  staging/invalid/
    (empty — no records

**Explanation:**  
- **Valid records:** Pass all checks — proceed to transformation.  
- **Invalid records:** Have NULL in a key column. These are quarantined — never loaded to OLTP/OLAP.  
- **Duplicate records:** Secondary occurrences of the same primary key.  
- **Outlier records:** Statistically unusual values (IQR method). Kept in valid set but flagged for review.  
- All outputs are saved as Parquet for fast downstream reading.


---
## Section 3 — Data Transformation

**Objective:**  
Apply business rules and data cleaning to each table:

| Table | Key Transformations |
|-------|--------------------|
| Product | Type casting, Color standardization, Date parsing, Price flag |
| Customer | AccountNumber normalization, NULL fill for optional columns |
| Sales | Join Header + Detail on SalesOrderID, select required columns |


In [6]:
print('Running transformation pipeline ...\n')
transformed = transform_all(val_result['valid'])

Running transformation pipeline ...


  Transforming Product ...
    Product: 504 clean records  (dropped 0 null-ID, 0 duplicates)
  Transforming Customer ...
    Customer: 19,820 clean records  (dropped 0 null-ID, 0 duplicates)
  Transforming Sales (join Header + Detail) ...
    Sales (joined): 121,317 line-item records
  Building aggregations ...

  Transformation complete:
    Product records    : 504
    Customer records   : 19,820
    Sales records      : 121,317
    Aggregation tables : 7


In [7]:
# Inspect transformed Product
df_prod = transformed['product']
print('── Transformed Product — Schema ──')
data_cols = [c for c in df_prod.columns if not c.startswith('_')]
print(df_prod[data_cols].dtypes.to_string())

print('\n── Sample (first 5 rows) ──')
display(df_prod[['ProductID','Name','Color','StandardCost','ListPrice','SellStartDate']].head())

print(f'\nUnique Colors in cleaned Product:')
print(df_prod['Color'].value_counts().to_string())

── Transformed Product — Schema ──
ProductID                         int64
Name                             object
ProductNumber                    object
MakeFlag                          int64
FinishedGoodsFlag                 int64
Color                            object
SafetyStockLevel                  int64
ReorderPoint                      int64
StandardCost                    float64
ListPrice                       float64
Size                             object
SizeUnitMeasureCode              object
WeightUnitMeasureCode            object
Weight                          float64
DaysToManufacture                 int64
ProductLine                      object
Class                            object
Style                            object
ProductSubcategoryID            float64
ProductModelID                  float64
SellStartDate            datetime64[ns]
SellEndDate              datetime64[ns]
DiscontinuedDate         datetime64[ns]
rowguid                          object
Modif

,ProductID,Name,Color,StandardCost,ListPrice,SellStartDate
0,1,Adjustable Race,N/A,0.00,0.00,2019-04-30
1,2,Bearing Ball,N/A,0.00,0.00,2019-04-30
2,3,BB Ball Bearing,N/A,0.00,0.00,2019-04-30
3,4,Headset Ball Bearings,N/A,0.00,0.00,2019-04-30
4,316,Blade,N/A,0.00,0.00,2019-04-30



Unique Colors in cleaned Product:
Color
N/A             248
Black            93
Silver           43
Red              38
Yellow           36
Blue             26
Multi             8
Silver/Black      7
White             4
Grey              1


In [8]:
# Inspect transformed Sales (joined)
df_sales = transformed['sales']
print('── Transformed Sales (Header + Detail joined) ──')
print(f'Rows  : {len(df_sales):,}')
print(f'Columns: {len(df_sales.columns)}')
print()
display(df_sales[[
    'SalesOrderID','SalesOrderDetailID','ProductID','CustomerID',
    'OrderDate','OrderQty','UnitPrice','LineTotal'
]].head())

── Transformed Sales (Header + Detail joined) ──
Rows  : 121,317
Columns: 18



,SalesOrderID,SalesOrderDetailID,ProductID,CustomerID,OrderDate,OrderQty,UnitPrice,LineTotal
0,43659,1,776,29825,2022-05-30,1,"2,024.99","2,024.99"
1,43659,2,777,29825,2022-05-30,3,"2,024.99","6,074.98"
2,43659,3,778,29825,2022-05-30,1,"2,024.99","2,024.99"
3,43659,4,771,29825,2022-05-30,1,"2,039.99","2,039.99"
4,43659,5,772,29825,2022-05-30,1,"2,039.99","2,039.99"


**Explanation:**  
- `StandardCost` and `ListPrice` are now `float64` — ready for arithmetic.  
- `SellStartDate` is now a proper `datetime64` — ready for time-series operations.  
- `Color` is title-cased and blank values replaced with `'N/A'` — standardized for grouping.  
- The Sales table is a **denormalized join** of Header + Detail — one row per line item,  
  combining order-level data (CustomerID, OrderDate) with line-level data (ProductID, OrderQty).


---
## Section 4 — Aggregations

**Objective:**  
Build summary datasets for analytics: top products, customer spending, sales trends.


In [9]:
aggs = transformed['aggregations']

print('── Top 10 Products by Sales ──')
top10 = aggs['top10_products'][['Name','Color','TotalSales','TotalQty','OrderCount']]
top10['TotalSales'] = top10['TotalSales'].map('${:,.0f}'.format)
top10['TotalQty']   = top10['TotalQty'].map('{:,}'.format)
display(top10)

── Top 10 Products by Sales ──


C:\Users\KEVIN\AppData\Local\Temp\ipykernel_20420\2387299962.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top10['TotalSales'] = top10['TotalSales'].map('${:,.0f}'.format)
C:\Users\KEVIN\AppData\Local\Temp\ipykernel_20420\2387299962.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top10['TotalQty']   = top10['TotalQty'].map('{:,}'.format)


,Name,Color,TotalSales,TotalQty,OrderCount
0,"Mountain-200 Black, 38",Black,"$4,400,593","2,977",1252
1,"Mountain-200 Black, 42",Black,"$4,009,495","2,664",1177
2,"Mountain-200 Silver, 38",Silver,"$3,693,678","2,394",1094
3,"Mountain-200 Silver, 42",Silver,"$3,438,479","2,234",1040
4,"Mountain-200 Silver, 46",Silver,"$3,434,257","2,216",1054
5,"Mountain-200 Black, 46",Black,"$3,309,673","2,111",1059
6,"Road-250 Black, 44",Black,"$2,516,857","1,642",705
7,"Road-250 Black, 48",Black,"$2,347,656","1,498",712
8,"Road-250 Black, 52",Black,"$2,012,448","1,245",667
9,"Road-150 Red, 56",Red,"$1,847,819",664,475


In [10]:
print('── Sales by Year ──')
yr = aggs['sales_by_year'].copy()
yr['TotalSales'] = yr['TotalSales'].map('${:,.0f}'.format)
display(yr)

── Sales by Year ──


,Year,TotalSales
0,2022,"$14,561,052"
1,2023,"$31,604,922"
2,2024,"$43,671,890"
3,2025,"$20,008,518"


In [11]:
print('── Sales by Month (most recent 12 months) ──')
mo = aggs['sales_by_month'].tail(12).copy()
mo['TotalSales'] = mo['TotalSales'].map('${:,.0f}'.format)
display(mo[['YearMonth','TotalSales']])

── Sales by Month (most recent 12 months) ──


,YearMonth,TotalSales
26,2024-07,"$4,918,580"
27,2024-08,"$3,325,296"
28,2024-09,"$4,539,829"
29,2024-10,"$4,820,725"
30,2024-11,"$3,291,238"
31,2024-12,"$4,082,854"
32,2025-01,"$4,276,428"
33,2025-02,"$3,565,879"
34,2025-03,"$4,987,902"
35,2025-04,"$5,222,759"


In [12]:
print('── Top 10 Customers by Sales ──')
top_cust = aggs['sales_by_customer'].head(10).copy()
top_cust['TotalSales'] = top_cust['TotalSales'].map('${:,.0f}'.format)
display(top_cust[['CustomerID','TotalSales','TotalQty','OrderCount']])

── Top 10 Customers by Sales ──


,CustomerID,TotalSales,TotalQty,OrderCount
18818,29818,"$877,107",1558,12
18715,29715,"$853,849",1322,12
18722,29722,"$841,909",2737,12
19117,30117,"$816,756",1736,12
18614,29614,"$799,278",1931,12
18639,29639,"$787,773",1708,12
18701,29701,"$746,318",1688,8
18617,29617,"$740,986",1344,12
18994,29994,"$730,799",1408,12
18646,29646,"$727,273",1344,12


**Explanation:**  
- `groupby().agg()` is the Pandas way to compute GROUP BY aggregations — equivalent to SQL `GROUP BY`.  
- We compute TotalSales (SUM), TotalQty (SUM), OrderCount (COUNT DISTINCT) per dimension.  
- Aggregations are saved as Parquet in `staging/valid/` for reuse by the dashboard notebook.  
- Top 10 by sales is the most commonly requested KPI in any retail analytics project.


---
## Section 5 — OLTP Schema Design

**Objective:**  
Understand the OLTP (Online Transaction Processing) schema and compare it to OLAP.

### What is OLTP?

| Characteristic | OLTP |
|---------------|------|
| Purpose | Record business transactions as they happen |
| Design | Normalized (3NF) — minimize data redundancy |
| Operations | Frequent INSERT, UPDATE, DELETE |
| Query type | Single-record lookups, small result sets |
| Joins | Many (normalized tables reference each other) |
| Example | A customer places an order → rows written to 2 tables |

### OLTP Relationships

```
product ←──────────────── salesorderdetail
                                │
customer ←── salesorderheader ──┘
```

**Foreign Key Rules:**
- Every `SalesOrderDetail.SalesOrderID` must exist in `SalesOrderHeader`
- Every `SalesOrderDetail.ProductID` must exist in `Product`
- Every `SalesOrderHeader.CustomerID` must exist in `Customer`


In [13]:
# Demonstrate the OLTP schema using our transformed DataFrames
print('OLTP Schema — Record Counts (what would be loaded to PostgreSQL)\n')
print('  Table                  Records    Primary Key')
print('  ' + '-' * 55)
print(f'  product                {len(transformed["product"]):>7,}    ProductID')
print(f'  customer               {len(transformed["customer"]):>7,}    CustomerID')

# SalesOrderHeader: load from valid staging (before join)
hdr_path = STAGING_VALID / 'valid_salesorderheader.parquet'
if hdr_path.exists():
    df_hdr = pd.read_parquet(hdr_path)
    print(f'  salesorderheader       {len(df_hdr):>7,}    SalesOrderID')

dtl_path = STAGING_VALID / 'valid_salesorderdetail.parquet'
if dtl_path.exists():
    df_dtl = pd.read_parquet(dtl_path)
    print(f'  salesorderdetail       {len(df_dtl):>7,}    SalesOrderID + SalesOrderDetailID')

print()
print('  SQL DDL: see sql/oltp_schema.sql')

OLTP Schema — Record Counts (what would be loaded to PostgreSQL)

  Table                  Records    Primary Key
  -------------------------------------------------------
  product                    504    ProductID
  customer                19,820    CustomerID
  salesorderheader        31,465    SalesOrderID
  salesorderdetail       121,317    SalesOrderID + SalesOrderDetailID

  SQL DDL: see sql/oltp_schema.sql


In [14]:
# Demonstrate referential integrity check (FK validation)
print('── Referential Integrity Check ──')
print('(Simulating what the DB FOREIGN KEY constraint enforces)\n')

df_sales = transformed['sales']
valid_product_ids  = set(transformed['product']['ProductID'].dropna().astype(int))
valid_customer_ids = set(transformed['customer']['CustomerID'].dropna().astype(int))

orphan_product  = df_sales[~df_sales['ProductID'].isin(valid_product_ids)]
orphan_customer = df_sales[~df_sales['CustomerID'].isin(valid_customer_ids)]

print(f'  Sales rows with invalid ProductID  : {len(orphan_product):,}')
print(f'  Sales rows with invalid CustomerID : {len(orphan_customer):,}')

if len(orphan_product) == 0 and len(orphan_customer) == 0:
    print('\n  All foreign key references are valid.')
else:
    print('\n  FK violations detected — these rows would be rejected by the DB.')

── Referential Integrity Check ──
(Simulating what the DB FOREIGN KEY constraint enforces)

  Sales rows with invalid ProductID  : 0
  Sales rows with invalid CustomerID : 0

  All foreign key references are valid.


**Explanation:**  
- In OLTP, **foreign keys** enforce data integrity at the database level.  
  A detail row cannot reference a product or order that doesn't exist.  
- We simulate this check in Pandas by comparing sets of IDs.  
- In `src/oltp.py`, the actual PostgreSQL tables enforce these constraints via `FOREIGN KEY` DDL.  
- The SQL DDL is in [`sql/oltp_schema.sql`](../sql/oltp_schema.sql).


---
## Section 6 — Star Schema (OLAP)

**Objective:**  
Build the Star Schema and understand the difference from OLTP.

### What is OLAP?

| Characteristic | OLAP |
|---------------|------|
| Purpose | Analyze large volumes of historical data |
| Design | Denormalized — fewer joins, faster aggregations |
| Operations | SELECT, GROUP BY, aggregations (no writes during analysis) |
| Query type | Large result sets, many rows |
| Joins | Few (fact → dimensions) |
| Example | "What were total sales per month per product category last year?" |

### Star Schema Structure

```
             DimDate
               │
DimProduct ──── FactSales ──── DimCustomer
```

- **Fact table** = what happened (one row per line item, numeric measures)
- **Dimension tables** = the context (who, what, when, where)
- **Surrogate keys** = integer keys generated by the DW (not from source system)


In [15]:
print('Building Star Schema ...\n')
olap_data = build_olap(transformed)

Building Star Schema ...

  Building DimDate ...
    DimDate: 2,922 rows
  Building DimProduct ...
    DimProduct: 504 rows
  Building DimCustomer ...
    DimCustomer: 19,820 rows
  Building FactSales ...
    FactSales: 121,317 rows


In [16]:
print('── DimDate — First 5 rows ──')
display(olap_data['dim_date'].head())
print(f'\nDimDate total rows: {len(olap_data["dim_date"]):,}')
print('Notice: Pre-computed Year, Quarter, Month, Week columns.')
print('These enable roll-up and drill-down without string parsing at query time.')

── DimDate — First 5 rows ──


,full_date,date_key,day_of_month,day_of_week,day_name,week_of_year,month_number,month_name,quarter,year,year_month,year_quarter,is_weekend,is_weekday
0,2019-01-01,20190101,1,2,Tuesday,1,1,January,1,2019,2019-01,2019-Q1,False,True
1,2019-01-02,20190102,2,3,Wednesday,1,1,January,1,2019,2019-01,2019-Q1,False,True
2,2019-01-03,20190103,3,4,Thursday,1,1,January,1,2019,2019-01,2019-Q1,False,True
3,2019-01-04,20190104,4,5,Friday,1,1,January,1,2019,2019-01,2019-Q1,False,True
4,2019-01-05,20190105,5,6,Saturday,1,1,January,1,2019,2019-01,2019-Q1,True,False



DimDate total rows: 2,922
Notice: Pre-computed Year, Quarter, Month, Week columns.
These enable roll-up and drill-down without string parsing at query time.


In [17]:
print('── DimProduct — First 5 rows ──')
display(olap_data['dim_product'][[
    'product_key','product_id','name','color','product_line',
    'standard_cost','list_price'
]].head())
print(f'\nDimProduct: {len(olap_data["dim_product"]):,} products')
print('product_key = surrogate key (generated by DW)')
print('product_id  = natural key  (from source CSV)')

── DimProduct — First 5 rows ──


,product_key,product_id,name,color,product_line,standard_cost,list_price
0,1,1,Adjustable Race,N/A,NaN,0.00,0.00
1,2,2,Bearing Ball,N/A,NaN,0.00,0.00
2,3,3,BB Ball Bearing,N/A,NaN,0.00,0.00
3,4,4,Headset Ball Bearings,N/A,NaN,0.00,0.00
4,5,316,Blade,N/A,NaN,0.00,0.00



DimProduct: 504 products
product_key = surrogate key (generated by DW)
product_id  = natural key  (from source CSV)


In [18]:
print('── FactSales — First 5 rows ──')
display(olap_data['fact_sales'][[
    'fact_id','product_key','customer_key','date_key',
    'sales_order_id','order_qty','unit_price','line_total'
]].head())

fact = olap_data['fact_sales']
print(f'\nFactSales: {len(fact):,} rows')
print(f'Total Sales Value: ${fact["line_total"].sum():,.2f}')
print(f'Total Units Sold : {fact["order_qty"].sum():,}')
print(f'Unique Orders    : {fact["sales_order_id"].nunique():,}')

── FactSales — First 5 rows ──


,fact_id,product_key,customer_key,date_key,sales_order_id,order_qty,unit_price,line_total
0,1,281,19527,20220530,43659,1,"2,024.99","2,024.99"
1,2,282,19527,20220530,43659,3,"2,024.99","6,074.98"
2,3,283,19527,20220530,43659,1,"2,024.99","2,024.99"
3,4,276,19527,20220530,43659,1,"2,039.99","2,039.99"
4,5,277,19527,20220530,43659,1,"2,039.99","2,039.99"



FactSales: 121,317 rows
Total Sales Value: $109,846,381.40
Total Units Sold : 274,914
Unique Orders    : 31,465


In [19]:
# OLTP vs OLAP comparison
print('\n── OLTP vs OLAP Comparison ──\n')
comparison = {
    'Model':           ['OLTP (Normalized 3NF)', 'OLAP (Star Schema)'],
    'Purpose':         ['Record transactions', 'Analyze history'],
    'Tables':          ['4 (product, customer, header, detail)', '4 (DimDate, DimProduct, DimCustomer, FactSales)'],
    'Joins needed':    ['3 (header-detail, detail-product, header-customer)', '1–2 (fact-dimension)'],
    'Key type':        ['Natural key (ProductID from source)', 'Surrogate key (product_key from DW)'],
    'Optimized for':   ['INSERT/UPDATE/DELETE', 'SELECT/GROUP BY/aggregations'],
}
display(pd.DataFrame(comparison).set_index('Model').T)


── OLTP vs OLAP Comparison ──



Model,OLTP (Normalized 3NF),OLAP (Star Schema)
Purpose,Record transactions,Analyze history
Tables,"4 (product, customer, header, detail)","4 (DimDate, DimProduct, DimCustomer, FactSales)"
Joins needed,"3 (header-detail, detail-product, header-custo...",1–2 (fact-dimension)
Key type,Natural key (ProductID from source),Surrogate key (product_key from DW)
Optimized for,INSERT/UPDATE/DELETE,SELECT/GROUP BY/aggregations


**Explanation:**  
- **DimDate** is a pre-built calendar table. Instead of parsing dates at query time,  
  all date attributes (month, quarter, year) are pre-computed.  
- **Surrogate keys** (`product_key`, `customer_key`) are integers generated by the data warehouse.  
  They are stable even if the source `ProductID` changes.  
- **FactSales** contains only numbers (measures) and foreign keys (surrogate keys).  
  No descriptive text — that lives in the dimension tables.  
- The SQL DDL is in [`sql/star_schema.sql`](../sql/star_schema.sql).


---
## Section 7 — OLAP Operations

**Objective:**  
Demonstrate the four classic OLAP operations using Pandas.

| Operation | Description | Example |
|-----------|-------------|--------|
| **Roll-up** | Aggregate to coarser granularity | Daily → Monthly → Yearly |
| **Drill-down** | Expand to finer granularity | Year → Quarter → Month → Day |
| **Slice** | Filter one dimension to a single value | Sales where color = 'Black' |
| **Dice** | Filter multiple dimensions simultaneously | Black/Silver products, 2022–2024, online orders |


In [20]:
fact = olap_data['fact_sales']
dim_date = olap_data['dim_date']
dim_product = olap_data['dim_product']
dim_customer = olap_data['dim_customer']

rollup = olap_rollup(fact, dim_date)

print('OLAP OPERATION 1: ROLL-UP')
print('Pattern: Daily → Monthly → Yearly\n')

print('  Yearly Sales (most coarse — highest level):')
yr = rollup['yearly'].copy()
yr['TotalSales'] = yr['TotalSales'].map('${:,.0f}'.format)
display(yr)

print('\n  Monthly Sales (sample — most recent 8 months):')
mo = rollup['monthly'].tail(8).copy()
mo['TotalSales'] = mo['TotalSales'].map('${:,.0f}'.format)
display(mo)

OLAP OPERATION 1: ROLL-UP
Pattern: Daily → Monthly → Yearly

  Yearly Sales (most coarse — highest level):


,year,TotalSales,TotalQty,OrderCount,UniqueCustomers
0,2022,"$14,561,052",15026,1692,1412
1,2023,"$31,604,922",66441,3830,3155
2,2024,"$43,671,890",131936,14244,11145
3,2025,"$20,008,518",61511,11699,10303



  Monthly Sales (sample — most recent 8 months):


,year_month,TotalSales,TotalQty,OrderCount
30,2024-11,"$3,291,238",9627,2100
31,2024-12,"$4,082,854",11059,2053
32,2025-01,"$4,276,428",11464,2136
33,2025-02,"$3,565,879",11436,1857
34,2025-03,"$4,987,902",15442,2304
35,2025-04,"$5,222,759",15539,2279
36,2025-05,"$1,908,059",5583,2218
37,2025-06,"$47,492",2047,905


In [21]:
drilldown = olap_drilldown(fact, dim_date)

print('OLAP OPERATION 2: DRILL-DOWN')
print('Pattern: Year → Quarter → Month → Day\n')

print('  Year level:')
display(drilldown['year'])

print('\n  Quarter level (first 8 rows):')
display(drilldown['quarter'].head(8))

print('\n  Explanation:')
print('  Drill-down lets analysts start with the big picture (year)')
print('  and "zoom in" to find the root cause of a trend.')

OLAP OPERATION 2: DRILL-DOWN
Pattern: Year → Quarter → Month → Day

  Year level:


,year,TotalSales
0,2022,"14,561,051.59"
1,2023,"31,604,921.95"
2,2024,"43,671,889.50"
3,2025,"20,008,518.36"



  Quarter level (first 8 rows):


,year,year_quarter,quarter,TotalSales
0,2022,2022-Q2,2,"2,519,016.40"
1,2022,2022-Q3,3,"5,831,058.22"
2,2022,2022-Q4,4,"6,210,976.97"
3,2023,2023-Q1,1,"6,502,423.05"
4,2023,2023-Q2,2,"8,808,557.97"
5,2023,2023-Q3,3,"9,047,743.03"
6,2023,2023-Q4,4,"7,246,197.90"
7,2024,2024-Q1,1,"7,838,170.11"



  Explanation:
  Drill-down lets analysts start with the big picture (year)
  and "zoom in" to find the root cause of a trend.


In [22]:
sliced = olap_slice(fact, dim_product, dim_date, color_filter='Black')

print('OLAP OPERATION 3: SLICE')
print('Filter: Color = "Black" only\n')
print('  Monthly sales for Black products:')
sliced_display = sliced.copy()
sliced_display['TotalSales'] = sliced_display['TotalSales'].map('${:,.0f}'.format)
display(sliced_display)

print('\n  Explanation:')
print('  Slice fixes ONE dimension to a single value.')
print('  Like cutting a slice of bread — you see one cross-section of the data cube.')

OLAP OPERATION 3: SLICE
Filter: Color = "Black" only

  Monthly sales for Black products:


,year_month,TotalSales,TotalQty
0,2022-05,"$212,664",226
1,2022-06,"$694,924",607
2,2022-07,"$528,801",458
3,2022-08,"$323,759",378
4,2022-09,"$880,384",871
5,2022-10,"$687,144",642
6,2022-11,"$331,236",292
7,2022-12,"$740,933",642
8,2023-01,"$598,270",523
9,2023-02,"$352,188",371



  Explanation:
  Slice fixes ONE dimension to a single value.
  Like cutting a slice of bread — you see one cross-section of the data cube.


In [23]:
diced = olap_dice(
    fact, dim_product, dim_customer, dim_date,
    colors=['Black', 'Silver'],
    year_range=(2022, 2024),
    online_only=False,
)

print('OLAP OPERATION 4: DICE')
print('Filters: color IN (Black, Silver) AND year BETWEEN 2022-2024\n')
print('  Top 10 product-month combinations:')
diced_display = diced.head(10).copy()
diced_display['TotalSales'] = diced_display['TotalSales'].map('${:,.0f}'.format)
display(diced_display)

print('\n  Explanation:')
print('  Dice applies filters on MULTIPLE dimensions simultaneously.')
print('  Like cutting a small cube out of the larger data cube.')

OLAP OPERATION 4: DICE
Filters: color IN (Black, Silver) AND year BETWEEN 2022-2024

  Top 10 product-month combinations:


,name,color,year_month,TotalSales,TotalQty
823,"Mountain-200 Black, 38",Black,2024-06,"$275,615",182
847,"Mountain-200 Black, 42",Black,2024-10,"$240,974",149
824,"Mountain-200 Black, 38",Black,2024-07,"$228,359",158
826,"Mountain-200 Black, 38",Black,2024-09,"$219,487",150
827,"Mountain-200 Black, 38",Black,2024-10,"$214,811",136
829,"Mountain-200 Black, 38",Black,2024-12,"$214,352",129
775,"Mountain-100 Silver, 42",Silver,2022-09,"$211,479",99
834,"Mountain-200 Black, 42",Black,2023-09,"$202,861",157
846,"Mountain-200 Black, 42",Black,2024-09,"$200,582",129
736,"Mountain-100 Black, 44",Black,2022-09,"$195,145",95



  Explanation:
  Dice applies filters on MULTIPLE dimensions simultaneously.
  Like cutting a small cube out of the larger data cube.


In [24]:
# Show the equivalent SQL for each operation
print('== Equivalent SQL Operations ==')
print('(See sql/analytics.sql for the full SQL versions)\n')

print('ROLL-UP (Daily -> Monthly):')
print('''
  SELECT d.year_month, SUM(f.line_total) AS monthly_sales
  FROM olap.fact_sales f
  JOIN olap.dim_date   d ON f.date_key = d.date_key
  GROUP BY d.year_month
  ORDER BY d.year_month;
''')

print('SLICE (color = Black):')
print('''
  SELECT d.year_month, SUM(f.line_total) AS total_sales
  FROM olap.fact_sales  f
  JOIN olap.dim_product  p ON f.product_key = p.product_key
  JOIN olap.dim_date     d ON f.date_key    = d.date_key
  WHERE p.color = 'Black'
  GROUP BY d.year_month;
''')

== Equivalent SQL Operations ==
(See sql/analytics.sql for the full SQL versions)

ROLL-UP (Daily -> Monthly):

  SELECT d.year_month, SUM(f.line_total) AS monthly_sales
  FROM olap.fact_sales f
  JOIN olap.dim_date   d ON f.date_key = d.date_key
  GROUP BY d.year_month
  ORDER BY d.year_month;

SLICE (color = Black):

  SELECT d.year_month, SUM(f.line_total) AS total_sales
  FROM olap.fact_sales  f
  JOIN olap.dim_product  p ON f.product_key = p.product_key
  JOIN olap.dim_date     d ON f.date_key    = d.date_key
  WHERE p.color = 'Black'
  GROUP BY d.year_month;



**Explanation:**  
- **Roll-up** reduces detail: `daily → monthly → yearly`. Each level is an aggregation of the one below it.  
- **Drill-down** is the reverse: start coarse, go finer. Analysts use this to investigate anomalies.  
- **Slice** is a single-value filter on one dimension (like SQL `WHERE color = 'Black'`).  
- **Dice** combines multiple filters (like SQL `WHERE color IN ('Black','Silver') AND year BETWEEN ...`).  
- All four operations are enabled by the Star Schema structure: the DimDate pre-computed hierarchy  
  makes roll-up/drill-down trivial, and dimension attributes make slicing/dicing fast.


---
## Section 8 — Step 2 Summary


In [25]:
fact = olap_data['fact_sales']
val_rep = val_result['reports']

total_outliers = sum(r['outliers']['outlier_records'] for r in val_rep.values())
total_invalid  = sum(r['nulls']['null_records'] for r in val_rep.values())
total_dupes    = sum(r['duplicates']['duplicate_records'] for r in val_rep.values())

print('=' * 60)
print('  STEP 2 COMPLETE — SCHEMA DESIGN SUMMARY')
print('=' * 60)
print()
print('  VALIDATION RESULTS:')
print(f'    Total source records : {sum(len(df) for df in raw.values()):>10,}')
print(f'    Invalid (null key)   : {total_invalid:>10,}')
print(f'    Duplicate records    : {total_dupes:>10,}')
print(f'    Outlier records      : {total_outliers:>10,}')
print()
print('  TRANSFORMATION RESULTS:')
print(f'    Product records      : {len(transformed["product"]):>10,}')
print(f'    Customer records     : {len(transformed["customer"]):>10,}')
print(f'    Sales records        : {len(transformed["sales"]):>10,}')
print()
print('  STAR SCHEMA:')
print(f'    DimDate rows         : {len(olap_data["dim_date"]):>10,}')
print(f'    DimProduct rows      : {len(olap_data["dim_product"]):>10,}')
print(f'    DimCustomer rows     : {len(olap_data["dim_customer"]):>10,}')
print(f'    FactSales rows       : {len(fact):>10,}')
print(f'    Total Sales Value    : ${fact["line_total"].sum():>19,.2f}')
print()
print('  SQL FILES GENERATED:')
print('    sql/oltp_schema.sql  — OLTP DDL (product, customer, orders)')
print('    sql/star_schema.sql  — Star Schema DDL (DimDate, DimProduct, DimCustomer, FactSales)')
print('    sql/analytics.sql    — OLAP operations (Roll-up, Drill-down, Slice, Dice)')
print()
print('  Next: Step 3 — Python Batch Pipeline (run_pipeline.py)')
print('=' * 60)

  STEP 2 COMPLETE — SCHEMA DESIGN SUMMARY

  VALIDATION RESULTS:
    Total source records :    173,106
    Invalid (null key)   :          0
    Duplicate records    :          0
    Outlier records      :     18,936

  TRANSFORMATION RESULTS:
    Product records      :        504
    Customer records     :     19,820
    Sales records        :    121,317

  STAR SCHEMA:
    DimDate rows         :      2,922
    DimProduct rows      :        504
    DimCustomer rows     :     19,820
    FactSales rows       :    121,317
    Total Sales Value    : $     109,846,381.40

  SQL FILES GENERATED:
    sql/oltp_schema.sql  — OLTP DDL (product, customer, orders)
    sql/star_schema.sql  — Star Schema DDL (DimDate, DimProduct, DimCustomer, FactSales)
    sql/analytics.sql    — OLAP operations (Roll-up, Drill-down, Slice, Dice)

  Next: Step 3 — Python Batch Pipeline (run_pipeline.py)


---
## Concepts Learned in This Notebook

| Concept | Definition |
|---------|------------|
| **Schema Validation** | Checking that expected columns are present before processing |
| **Null Validation** | Key columns must not be empty — records with null keys are quarantined |
| **IQR Outlier Detection** | `Q1 - 1.5×IQR` to `Q3 + 1.5×IQR` defines the normal range; values outside are outliers |
| **Data Transformation** | Type casting, string normalization, NULL filling, joining tables |
| **OLTP** | Normalized operational database — captures business transactions |
| **OLAP** | Denormalized analytical database — optimized for fast aggregations |
| **Star Schema** | One Fact table surrounded by Dimension tables |
| **Fact Table** | Stores numeric measures (line_total, order_qty) + surrogate foreign keys |
| **Dimension Table** | Stores descriptive attributes (product name, customer account, date hierarchy) |
| **Surrogate Key** | Integer key generated by the data warehouse (not from source) |
| **Roll-up** | Aggregating from fine grain to coarse grain (day → month → year) |
| **Drill-down** | Moving from coarse grain to fine grain (year → quarter → month → day) |
| **Slice** | Filter one dimension to a single value |
| **Dice** | Filter multiple dimensions simultaneously |

---
*Next notebook:* `03_batch_pipeline.ipynb` — Python Batch Pipeline with run_pipeline()  
*Developed with:* **Antigravity** — AI Coding Assistant
